# Windows UTC offset — GUM Warszawa + NIST (NTPv4)

**Cel:** odczytać, o ile zegar systemowy Windows wyprzedza / jest opóźniony względem referencyjnych serwerów czasu. **Kod nie zmienia zegara.** Uruchom kolejno komórki `Shift+Enter`, a w razie potrzeby zmień parametr `ROUNDS` w komórce kodu.

### Źródła

- **GUM Warszawa:** `tempus1.gum.gov.pl`, `tempus2.gum.gov.pl` — krajowy wzorzec czasu, [potwierdzenie GUM](https://www.gum.gov.pl/pl/dla-biznesu/uslugi/zegar/524%2CZegar.html).
- **NIST USA:** `time-a-g.nist.gov`, `time-d-b.nist.gov` — UTC(NIST), [oficjalny wykaz serwerów i limit częstotliwości](https://tf.nist.gov/tf-cgi/servers.cgi/en-en/).
- [RFC 5905 — algorytm NTP](https://www.rfc-editor.org/rfc/rfc5905), [Microsoft Win32 — precyzyjny czas UTC](https://learn.microsoft.com/en-us/windows/win32/api/sysinfoapi/nf-sysinfoapi-getsystemtimepreciseasfiletime).

### Znaczenie znaków i metryk

- **`PC−UTC > 0`**: zegar Windows **wyprzedza** wzorzec. **`PC−UTC < 0`**: zegar Windows jest **spóźniony**.
- `offset`: mediana najlepszych próbek, liczona z czterech stempli czasu NTP.
- `jitter`: rozrzut wyników (MAD × 1,4826), **nie dokładność bezwzględna**.
- `RTT/2`: skala maksymalnej nierozróżnialnej asymetrii sieciowej; przy zwykłym NTP przez Internet nie możemy udowodnić dokładności ±1 µs.
- `t1` i `t4` pochodzą z userspace — **nie są sprzętowymi timestampami NIC**.

**Środowisko:** lokalny kernel Python w Windows 11, działające DNS i wychodzący UDP/123. Nie uruchamiaj na zdalnym kernelu lub w WSL, jeśli badasz zegar Windows. Zewnętrzne biblioteki nie są wymagane do pomiaru. Czas domyślny około 1 minuty (zależny od sieci). Wyniki surowe zapisują się w CSV obok notebooka. Żeby nie przeciążać NIST, zapytania są rozdzielone co najmniej 4,5 sekundy.


In [ ]:
"""Pomiar offsetu zegara Windows wzgledem UTC(PL)/UTC(NIST), bez zmiany zegara.

W notebooku uruchom: %run utc_offset_ntp_windows.py
Uwaga: publiczny NTP nie umozliwia certyfikacji dokladnosci +/-1 us.
Bez zaleznosci poza biblioteka standardowa. Windows 8+ / Python 3.10+.
"""

from __future__ import annotations

import csv
import ctypes
import math
import platform
import socket
import statistics
import struct
import sys
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

# --- KONFIGURACJA ---
ROUNDS = 12               # 12 rund ~= 60-90 s; zwieksz do 30 dla dluzszej serii
TIMEOUT_S = 2.0           # deadline pojedynczego zapytania UDP
GUM_MIN_GAP_S = 4.5       # nie odpytuj danego serwera zbyt czesto
NIST_GLOBAL_GAP_S = 4.5  # >= 4 s miedzy zapytaniami do dowolnego NIST
GAP_BETWEEN_QUERIES_S = 0.25
OUTDIR = Path.cwd()

SERVERS = (
    {"name": "GUM-1", "host": "tempus1.gum.gov.pl", "group": "GUM"},
    {"name": "GUM-2", "host": "tempus2.gum.gov.pl", "group": "GUM"},
    {"name": "NIST-East", "host": "time-a-g.nist.gov", "group": "NIST"},
    {"name": "NIST-West", "host": "time-d-b.nist.gov", "group": "NIST"},
)

NTP_UNIX_DELTA_S = 2_208_988_800
NTP_ERA_S = 1 << 32
NANO = 1_000_000_000
WIN_FILETIME_TO_UNIX_100NS = 116_444_736_000_000_000

if sys.platform == "win32":
    class FILETIME(ctypes.Structure):
        _fields_ = [("dwLowDateTime", ctypes.c_uint32),
                    ("dwHighDateTime", ctypes.c_uint32)]

    _k32 = ctypes.WinDLL("kernel32", use_last_error=True)
    _precise = _k32.GetSystemTimePreciseAsFileTime
    _precise.argtypes = [ctypes.POINTER(FILETIME)]
    _precise.restype = None

    def system_utc_ns() -> int:
        """Win32 high precision UTC timestamp; rozdzielczosc != dokladnosc UTC."""
        ft = FILETIME()
        _precise(ctypes.byref(ft))
        ticks_100ns = (ft.dwHighDateTime << 32) | ft.dwLowDateTime
        return (ticks_100ns - WIN_FILETIME_TO_UNIX_100NS) * 100

else:
    # Umozliwia test jednostkowy w srodowisku nie-Windows; wlasciwy run blokowany.
    def system_utc_ns() -> int:
        return time.time_ns()


def cross_timestamp():
    """Para zegarow: monotoniczny QPC + Windows UTC, z szerokoscia odczytu."""
    m0 = time.perf_counter_ns()
    utc = system_utc_ns()
    m1 = time.perf_counter_ns()
    return utc, (m0 + m1) // 2, m1 - m0


def unix_to_ntp_stamp(unix_ns: int) -> int:
    """64-bit NTP seconds+fraction. Niezbedne pole originate w odpowiedzi."""
    sec, ns = divmod(unix_ns, NANO)
    ntp_sec = (sec + NTP_UNIX_DELTA_S) & 0xFFFFFFFF
    frac = (ns << 32) // NANO
    return (ntp_sec << 32) | frac


def ntp_stamp_to_unix_ns(stamp: int, reference_unix_ns: int) -> int:
    """Odtwarzanie ery NTP (w tym po 2036) na podstawie lokalnej epoki."""
    sec32, frac = stamp >> 32, stamp & 0xFFFFFFFF
    approx_ntp_sec = (reference_unix_ns // NANO) + NTP_UNIX_DELTA_S
    era = (approx_ntp_sec - sec32 + (NTP_ERA_S // 2)) // NTP_ERA_S
    full_sec = era * NTP_ERA_S + sec32
    frac_ns = (frac * NANO + (1 << 31)) >> 32
    return (full_sec - NTP_UNIX_DELTA_S) * NANO + frac_ns


def resolve_ipv4(hostname: str) -> str:
    """Jedno przypisane IP na caly pomiar, bez rotacji DNS miedzy probkami."""
    info = socket.getaddrinfo(hostname, 123, socket.AF_INET, socket.SOCK_DGRAM)
    if not info:
        raise OSError("DNS nie zwrocil adresu IPv4")
    return info[0][4][0]


def ntp_probe(name: str, group: str, ip: str, index: int, timeout_s=TIMEOUT_S, port: int = 123):
    row = {
        "round": index, "server": name, "group": group, "ip": ip,
        "status": "ERROR", "error": "", "t1_pc_ns": "", "t2_server_ns": "",
        "t3_server_ns": "", "t4_pc_ns": "", "qpc_t1_ns": "",
        "qpc_t4_ns": "", "span_t1_ns": "", "span_t4_ns": "",
        "theta_utc_minus_pc_ns": "", "pc_minus_utc_ns": "",
        "ntp_delay_ns": "", "qpc_rtt_ns": "", "clock_interval_discrepancy_ns": "",
        "root_delay_ms": "", "root_disp_ms": "", "stratum": "",
        "leap": "", "ntp_version": "", "peer_refid": "", "request_cookie": "",
    }
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as sock:
            sock.settimeout(timeout_s)
            sock.connect((ip, port))

            # Przygotuj pakiet PRZED krytycznym pomiarem t1.
            request = bytearray(48)
            request[0] = (4 << 3) | 3  # NTPv4, client mode=3
            cookie = unix_to_ntp_stamp(system_utc_ns())
            request[40:48] = struct.pack("!Q", cookie)
            row["request_cookie"] = f"0x{cookie:016x}"

            # t1: odczyt zegara systemowego najblizej jak sie da do send().
            t1, qpc_t1, span1 = cross_timestamp()
            sent = sock.send(request)
            if sent != 48:
                raise OSError(f"Wyslano tylko {sent}/48 bajtow")
            response = sock.recv(512)
            # t4: odczyt NATYCHMIAST po wyjsciu z recv() (nie NIC timestamp).
            t4, qpc_t4, span4 = cross_timestamp()

        row.update({"t1_pc_ns": t1, "t4_pc_ns": t4,
                    "qpc_t1_ns": qpc_t1, "qpc_t4_ns": qpc_t4,
                    "span_t1_ns": span1, "span_t4_ns": span4})
        if len(response) < 48:
            raise ValueError(f"Zbyt krotka odpowiedz: {len(response)} B")

        li = response[0] >> 6
        version = (response[0] >> 3) & 0x7
        mode = response[0] & 0x7
        stratum = response[1]
        refid = response[12:16]
        row.update({"leap": li, "ntp_version": version, "stratum": stratum,
                    "peer_refid": refid.hex()})

        if mode != 4 or version not in (3, 4):
            raise ValueError(f"Nieprawidlowy NTP mode={mode}, version={version}")
        if stratum == 0:
            try:
                kod = refid.decode("ascii")
            except UnicodeDecodeError:
                kod = refid.hex()
            raise ValueError(f"KoD (Kiss-o'-Death): {kod}")
        if li == 3 or not (1 <= stratum <= 15):
            raise ValueError(f"Serwer niezsynch.: LI={li}, stratum={stratum}")

        origin = struct.unpack_from("!Q", response, 24)[0]
        rx_stamp = struct.unpack_from("!Q", response, 32)[0]
        tx_stamp = struct.unpack_from("!Q", response, 40)[0]
        if origin != cookie:
            raise ValueError("Originate timestamp != wyslane xmt (zly pakiet / replay)")
        if rx_stamp == 0 or tx_stamp == 0:
            raise ValueError("Brak znacznika czasu serwera t2/t3")

        t2 = ntp_stamp_to_unix_ns(rx_stamp, t1)
        t3 = ntp_stamp_to_unix_ns(tx_stamp, t4)
        if t3 < t2:
            raise ValueError("Ujemny czas przetwarzania serwera t3 < t2")

        # RFC 5905: theta=UTC_ref-PC, delay=RTT z odjeta praca serwera.
        theta_twice_ns = (t2 - t1) + (t3 - t4)
        theta_ns = theta_twice_ns / 2  # max 0.5ns ulamkowej rozdzielczosci
        delta_ns = (t4 - t1) - (t3 - t2)
        mono_rtt_ns = qpc_t4 - qpc_t1
        clock_discrepancy_ns = (t4 - t1) - mono_rtt_ns
        if delta_ns <= 0:
            raise ValueError(f"Niefizyczny delta={delta_ns}ns")
        # Wykrywaj skoki / duze korekty systemowego czasu miedzy t1 i t4.
        # Nie jest to kalibrowany prog nanosekundowy.
        if abs(clock_discrepancy_ns) > 500_000:
            raise ValueError(
                f"Podejrzenie skoku zegara w trakcie pakietu: "
                f"wall-QPC={clock_discrepancy_ns/1e6:.3f}ms"
            )

        root_delay_ms = struct.unpack_from("!i", response, 4)[0] / 65536 * 1000
        root_disp_ms = struct.unpack_from("!I", response, 8)[0] / 65536 * 1000
        row.update({
            "t2_server_ns": t2, "t3_server_ns": t3,
            "theta_utc_minus_pc_ns": theta_ns,
            "pc_minus_utc_ns": -theta_ns,
            "ntp_delay_ns": delta_ns, "qpc_rtt_ns": mono_rtt_ns,
            "clock_interval_discrepancy_ns": clock_discrepancy_ns,
            "root_delay_ms": root_delay_ms, "root_disp_ms": root_disp_ms,
            "status": "OK",
        })

    except (OSError, ValueError, struct.error) as exc:
        row["error"] = str(exc)

    return row


def clock_diagnostics(n: int = 2000):
    """Empiryczny krok kolejnych odczytow czasu, NIE dokladnosc UTC."""
    utc = [system_utc_ns() for _ in range(n)]
    mono = [time.perf_counter_ns() for _ in range(n)]
    du = [b-a for a,b in zip(utc,utc[1:]) if b > a]
    dm = [b-a for a,b in zip(mono,mono[1:]) if b > a]
    print("Probek rozdzielczosci:", n)
    print("Min krok dodatni Win UTC:", min(du) if du else "brak", "ns")
    print("Min krok dodatni perf_counter/QPC:", min(dm) if dm else "brak", "ns")
    print("Przypomnienie: min krok odczytu != dokladnosc UTC lub NIC timestamp.")


def theil_sen_clock_drift_ppm(selected_rows):
    """Drift PC wzgledem serwera z mediany nachylen par. Tylko diagnostyka."""
    if len(selected_rows) < 5:
        return None
    p = sorted((((r["qpc_t1_ns"] + r["qpc_t4_ns"]) // 2),
                r["theta_utc_minus_pc_ns"]) for r in selected_rows)
    if p[-1][0] - p[0][0] < 15 * NANO:
        return None
    slopes = []
    for i in range(len(p)):
        for j in range(i+1, len(p)):
            dt = p[j][0] - p[i][0]
            if dt >= 3 * NANO:
                slopes.append(-((p[j][1] - p[i][1]) / dt) * 1_000_000)
    return statistics.median(slopes) if slopes else None


def median_abs_dev(nums):
    center = statistics.median(nums)
    return statistics.median(abs(x - center) for x in nums)


def analyze_server(rows):
    valid = [r for r in rows if r["status"] == "OK"]
    if not valid:
        return None
    ordered = sorted(valid, key=lambda r: r["ntp_delay_ns"])
    min_rtt_ns = ordered[0]["ntp_delay_ns"]
    # Wybieraj probki o niskim RTT, aby ograniczyc dodatkowe kolejki.
    # Dynamiczne okno [0.25 .. 2.0] ms, ok. 5% min RTT.
    rtt_window_ns = max(250_000, min(2_000_000, int(0.05 * min_rtt_ns)))
    filtered = [r for r in ordered if r["ntp_delay_ns"] <= min_rtt_ns + rtt_window_ns]
    if len(filtered) < min(3, len(ordered)):
        filtered = ordered[:min(4, len(ordered))]

    offsets = [r["theta_utc_minus_pc_ns"] for r in filtered]
    median_theta_ns = statistics.median(offsets)
    mad_ns = median_abs_dev(offsets)
    robust_jitter_ns = 1.4826 * mad_ns
    # Wysoka rozdzielczosc != mniejsza niepewnosc bez kalibracji sieci.
    best = ordered[0]
    estimated_asymmetry_envelope_ns = min_rtt_ns / 2
    # Root delay/dispersion sa parametrami serwera, nie certyfikatem bledu.
    extra_server_budget_ns = (
        max(0.0, best["root_delay_ms"]) / 2 + best["root_disp_ms"]
    ) * 1e6
    # Przyczynowosciowy przedzial z jednej probki o min RTT.
    # PC-UTC moze byc w [t1-t2, t4-t3] przy nieujemnych opoznieniach.
    # Rozszerzamy heurystycznie o root delay/disp serwera.
    lower_bound_ns = best["t1_pc_ns"] - best["t2_server_ns"] - extra_server_budget_ns
    upper_bound_ns = best["t4_pc_ns"] - best["t3_server_ns"] + extra_server_budget_ns
    approximate_drift_ppm = theil_sen_clock_drift_ppm(filtered)

    return {
        "server": valid[0]["server"], "group": valid[0]["group"],
        "ip": valid[0]["ip"], "n": len(valid), "n_filtered": len(filtered),
        "utc_minus_pc_ns": median_theta_ns,
        "pc_minus_utc_ns": -median_theta_ns,
        "min_rtt_ms": min_rtt_ns / 1e6,
        "median_rtt_ms": statistics.median(r["ntp_delay_ns"] for r in valid) / 1e6,
        "robust_jitter_ms": robust_jitter_ns / 1e6,
        "root_delay_ms": best["root_delay_ms"],
        "root_disp_ms": best["root_disp_ms"],
        "unresolvable_asymmetry_ms": estimated_asymmetry_envelope_ns / 1e6,
        "root_path_budget_ms": extra_server_budget_ns / 1e6,
        "causality_pc_minus_utc_interval_ms": (lower_bound_ns/1e6, upper_bound_ns/1e6),
        "approximate_clock_drift_ppm": approximate_drift_ppm,
        "selected_rows": filtered,
    }


def run_measurement(rounds: int = ROUNDS, timeout_s: float = TIMEOUT_S,
                    allow_non_windows: bool = False):
    if sys.platform != "win32" and not allow_non_windows:
        raise RuntimeError(
            "Uruchom notebook na LOCALNYM kernelu Python Windows, a nie WSL / remote."
        )

    print("=== Windows clock -> NTP UTC offset measurement ===")
    print("System:", platform.platform())
    print("Zegar:", "Win32 GetSystemTimePreciseAsFileTime" if sys.platform == "win32"
          else "test fallback time.time_ns()")
    print("QPC:", time.get_clock_info("perf_counter"))
    clock_diagnostics()
    print("Uwaga: kod NIE modyfikuje zegara systemowego.")
    print("Uwaga: NIST wymaga odstepu co najmniej 4 s miedzy zapytaniami.\n")

    dns = {}
    for server in SERVERS:
        try:
            dns[server["name"]] = resolve_ipv4(server["host"])
            print(f"DNS {server['name']:<11} {server['host']:<25} -> {dns[server['name']]}")
        except OSError as exc:
            print(f"DNS BLAD {server['name']}: {exc}")

    if not dns:
        raise RuntimeError("Brak dostepnych serwerow DNS.")

    rows = []
    last_start = {}  # monotonic float start per hostname
    last_nist_start = None
    all_gum = [s for s in SERVERS if s["group"] == "GUM"]
    nists = [s for s in SERVERS if s["group"] == "NIST"]
    t_begin = time.perf_counter()
    try:
        for idx in range(1, rounds + 1):
            scheduled = all_gum + [nists[(idx - 1) % len(nists)]]
            for srv in scheduled:
                if srv["name"] not in dns:
                    continue
                host = srv["host"]
                now = time.perf_counter()
                wait_host = (
                    last_start.get(host, -1e50) + GUM_MIN_GAP_S - now
                )
                wait_nist = (
                    (last_nist_start or -1e50) + NIST_GLOBAL_GAP_S - now
                    if srv["group"] == "NIST" else 0
                )
                wait = max(0.0, wait_host, wait_nist)
                if wait:
                    time.sleep(wait)

                started = time.perf_counter()
                last_start[host] = started
                if srv["group"] == "NIST":
                    last_nist_start = started

                sample = ntp_probe(
                    srv["name"], srv["group"], dns[srv["name"]], idx,
                    timeout_s=timeout_s
                )
                rows.append(sample)
                if sample["status"] == "OK":
                    print(f"[{idx:02d}/{rounds}] {srv['name']:<11} "
                          f"PC-UTC={sample['pc_minus_utc_ns']/1e6:+10.4f} ms "
                          f"RTT={sample['ntp_delay_ns']/1e6:8.3f} ms "
                          f"stratum={sample['stratum']}")
                else:
                    print(f"[{idx:02d}/{rounds}] {srv['name']:<11} "
                          f"ERROR: {sample['error']}")
                time.sleep(GAP_BETWEEN_QUERIES_S)
    except KeyboardInterrupt:
        print("Pomiar przerwany. Zapisuje dotychczasowe dane.")

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    OUTDIR.mkdir(parents=True, exist_ok=True)
    csv_file = OUTDIR / f"utc_ntp_raw_{stamp}.csv"
    with csv_file.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else [])
        if rows:
            writer.writeheader()
            writer.writerows(rows)
    print(f"\nCSV raw: {csv_file}")
    print(f"Czas badania: {time.perf_counter()-t_begin:.1f}s; proby: {len(rows)}")

    grouped = defaultdict(list)
    for row in rows:
        grouped[row["server"]].append(row)
    reports = [analyze_server(grouped[srv["name"]]) for srv in SERVERS
               if grouped[srv["name"]]]
    reports = [r for r in reports if r is not None]
    if not reports:
        print("\nBRAK POMIAROW NTP! Sprawdz, czy UDP/123 jest przepuszczane.")
        return rows, []

    print("\n=== WYNIK PER SERWER (filtered = najnizsze RTT) ===")
    print("  PC-UTC > 0 = zegar PC wyprzedza UTC; PC-UTC < 0 = PC jest opozniony.")
    print(f"{'Serwer':<11} {'OK/filtr':>9} {'PC-UTC [ms]':>14} "
          f"{'minRTT [ms]':>12} {'jitter [ms]':>12} {'+/-RTT/2 [ms]':>15}")
    for r in reports:
        print(f"{r['server']:<11} {r['n']:>3}/{r['n_filtered']:<5} "
              f"{r['pc_minus_utc_ns']/1e6:+14.4f} "
              f"{r['min_rtt_ms']:12.3f} {r['robust_jitter_ms']:12.4f} "
              f"{r['unresolvable_asymmetry_ms']:15.3f}")

    gums = [r for r in reports if r["group"] == "GUM"]
    if gums:
        preferred = min(gums, key=lambda r: (r["min_rtt_ms"], r["robust_jitter_ms"]))
    else:
        preferred = min(reports, key=lambda r: (r["min_rtt_ms"], r["robust_jitter_ms"]))

    print("\n=== REFERENCJA O NAJMNIEJSZYM ZAOBSERWOWANYM RTT (priorytet GUM) ===")
    diff_ms = preferred["pc_minus_utc_ns"] / 1e6
    state = "PRZYSPIESZONY" if diff_ms > 0 else "OPOZNIONY" if diff_ms < 0 else "ZGODNY"
    print(f"Serwer: {preferred['server']} ({preferred['ip']})")
    print(f"Zegar PC wzgledem serwera UTC: {diff_ms:+.3f} ms ({state})")
    print(f"Szacowana korekta potrzebna do wyrownania PC: {-diff_ms:+.3f} ms")
    print(f"Jitter probek niskiego RTT (robust sigma): "
          f"{preferred['robust_jitter_ms']*1000:.1f} us")
    print(f"Nierozroznialna asymetria sieci bez kalibracji (do ok. RTT/2): "
          f"{preferred['unresolvable_asymmetry_ms']:.3f} ms")
    low, high = preferred["causality_pc_minus_utc_interval_ms"]
    print(f"Szeroki przedzial PC-UTC z najszybszego pakietu, z uwzglednieniem "
          f"metadanych serwera: [{low:+.3f}, {high:+.3f}] ms")
    print("Ten przedzial zaklada nieujemne opoznienia sieciowe i dobre znaczniki; "
          "NIE jest certyfikowana niepewnoscia.")
    if preferred["approximate_clock_drift_ppm"] is not None:
        print(f"Orientacyjna zmiana szybkosci zegara PC wzgledem UTC z obserwacji "
              f"(Theil-Sen): {preferred['approximate_clock_drift_ppm']:+.2f} ppm")
    print("To NIE JEST deklaracja dokladnosci UTC +/-1us ani kalibrowana niepewnosc.")

    if len(gums) >= 2:
        difference_ms = abs(gums[0]["pc_minus_utc_ns"] - gums[1]["pc_minus_utc_ns"]) / 1e6
        print(f"Roznica pomiedzy medianami GUM: {difference_ms:.4f} ms")
        if difference_ms > max(0.5, 3 * sum(r["robust_jitter_ms"] for r in gums)):
            print("UWAGA: GUM-1 i GUM-2 sa niezgodne. Podejrzewaj asymetrie lub zmiane zegara.")

    print("NIST sluzy jako niezalezny punkt kontrolny; jego RTT moze byc wielokrotnie wieksze.")
    print("Parametry root delay/root dispersion sa w CSV, w diagnostyce mozna je dodac do budzetu.")
    return rows, reports


## Wykonaj pomiary (kilkadziesiąt sekund)

W wynikach szukaj sekcji **REFERENCJA** i **WYNIK PER SERWER**. Jeżeli zobaczysz wyłącznie `ERROR` — najpierw sprawdź UDP/123 i firewall.

In [ ]:
raw_rows, server_reports = run_measurement(rounds=ROUNDS)

## Tabela podsumowania i wykresy (opcjonalne)

Wymaga `pandas`/`matplotlib`, jeśli nie masz ich jeszcze w środowisku. Gdy ich brakuje, sam pomiar i plik CSV nadal działają.

In [ ]:
try:
    import pandas as pd
    if server_reports:
        summary_table = pd.DataFrame([
            {
                "Serwer": r["server"],
                "IP": r["ip"],
                "Pomiary poprawne": r["n"],
                "Próbki niskiego RTT": r["n_filtered"],
                "PC - UTC [ms]": round(r["pc_minus_utc_ns"] / 1e6, 6),
                "min RTT [ms]": round(r["min_rtt_ms"], 3),
                "jitter MAD [µs]": round(r["robust_jitter_ms"] * 1e3, 2),
                "RTT/2 [ms]": round(r["unresolvable_asymmetry_ms"], 3),
                "dryft diagnostyczny [ppm]": r["approximate_clock_drift_ppm"],
            } for r in server_reports
        ])
        display(summary_table)
    else:
        print("Brak poprawnych próbek do tabeli.")
except ImportError:
    print("Pandas nie jest zainstalowany; podsumowanie tekstowe wyświetlono powyżej.")

try:
    import matplotlib.pyplot as plt
    ok = [r for r in raw_rows if r["status"] == "OK"]
    if ok:
        t0 = min(r["qpc_t1_ns"] for r in ok)

        plt.figure(figsize=(12, 4))
        for name in dict.fromkeys(r["server"] for r in ok):
            data = [r for r in ok if r["server"] == name]
            plt.scatter([(r["qpc_t1_ns"] - t0) / 1e9 for r in data],
                        [r["pc_minus_utc_ns"] / 1e6 for r in data],
                        label=name, s=18)
        plt.axhline(0, linestyle="--", linewidth=0.8)
        plt.xlabel("Czas od startu [s] (QPC)")
        plt.ylabel("PC - UTC [ms] (NTP)")
        plt.title("Offset zegara Windows od referencji GUM i NIST")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.show()

        plt.figure(figsize=(12, 4))
        for name in dict.fromkeys(r["server"] for r in ok):
            data = [r for r in ok if r["server"] == name]
            plt.scatter([(r["qpc_t1_ns"] - t0) / 1e9 for r in data],
                        [r["ntp_delay_ns"] / 1e6 for r in data],
                        label=name, s=18)
        plt.xlabel("Czas od startu [s] (QPC)")
        plt.ylabel("RTT netto [ms]")
        plt.title("RTT NTP — GUM (blisko) vs NIST (daleko)")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.show()
except ImportError:
    print("Matplotlib nie jest zainstalowany; mozna opcjonalnie: %pip install matplotlib")


In [ ]:

import asyncio
import time
import statistics
import websockets

URLS = {
    "SPOT": (
        "wss://stream.binance.com:443/stream"
        "?streams=btcusdt@trade&timeUnit=microsecond"
    ),
    "USD-M FUTURES": (
        "wss://fstream.binance.com/stream"
        "?streams=btcusdt@trade"
    ),
    "COIN-M FUTURES": (
        "wss://dstream.binance.com/stream"
        "?streams=btcusd_perp@trade"
    ),
}

async def measure_ping(market, url, n=5):
    try:
        async with websockets.connect(
            url,
            ping_interval=None,
            open_timeout=8,
            close_timeout=2,
        ) as ws:

            ip = ws.remote_address[0]
            rtts = []

            for _ in range(n):
                start = time.perf_counter_ns()
                pong = await ws.ping()
                await asyncio.wait_for(pong, timeout=5)

                rtt_ms = (
                    time.perf_counter_ns() - start
                ) / 1_000_000

                rtts.append(rtt_ms)
                await asyncio.sleep(0.1)

            return market, ip, rtts, None

    except Exception as exc:
        return market, None, [], str(exc)


results = await asyncio.gather(
    *(measure_ping(market, url) for market, url in URLS.items())
)

for market, ip, rtts, error in results:
    print(f"\n{'=' * 45}")
    print(f" {market}")
    print(f"{'=' * 45}")

    if error:
        print(f"ERROR: {error}")
        continue

    print(f"Server IP  : {ip}")
    print(f"RTT MIN    : {min(rtts):.3f} ms")
    print(f"RTT MEDIAN : {statistics.median(rtts):.3f} ms")
    print(f"RTT MAX    : {max(rtts):.3f} ms")
    print(f"RTT SAMPLES: {[round(x, 3) for x in rtts]}")
    print(f"ONE-WAY ~  : {statistics.median(rtts)/2:.3f} ms")
